# 2. Document Parsing

This notebook demonstrates two core parsing approaches:

1. **AI_PARSE** - Databricks native AI-powered document parsing (simplest)
2. **Docling Single** - Single document processing with native Docling (most flexible)

Both methods will populate the same Delta tables for use by the chat application.

In [0]:
%pip install "docling[easyocr]" docling_core tabulate typing_inspection pydantic_settings python-dotenv --no-deps
%pip install easyocr
%restart_python

In [0]:
from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PipelineOptions, EasyOcrOptions
from docling.datamodel.accelerator_options import AcceleratorOptions, AcceleratorDevice

pipeline_options = PipelineOptions()
pipeline_options.do_ocr = True
pipeline_options.ocr_options = EasyOcrOptions()  # or RapidOcrOptions(...)
pipeline_options.accelerator = AcceleratorOptions(
    device=AcceleratorDevice.CPU,  # force CPU
    num_threads=8,
)

converter = DocumentConverter(options=pipeline_options)

result = converter.convert("/Volumes/main/default/raw_docs/your_file.pdf")
print(result.status)

In [0]:
%pip install --upgrade "docling"

In [0]:
%pip install "torch==2.7.1" "torchvision==0.22.1" --no-deps

In [0]:
%restart_python

In [0]:
import pkg_resources

for dist in pkg_resources.working_set:
    for req in dist.requires():
        if "torch" in req.project_name.lower():
            print(dist.project_name, "requires", req)

In [0]:
%pip install uv
%uv sync
%restart_python

In [0]:
import pynvml
import psutil

# GPU memory (NVIDIA only)
try:
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    mem_info = pynvml.nvmlDeviceGetMemoryInfo(handle)
    gpu_total = mem_info.total / (1024 ** 3)
    gpu_free = mem_info.free / (1024 ** 3)
    print(f"GPU Memory: {gpu_free:.2f} GB free / {gpu_total:.2f} GB total")
    pynvml.nvmlShutdown()
except Exception as e:
    print(f"GPU info unavailable: {e}")

# RAM memory
ram = psutil.virtual_memory()
ram_total = ram.total / (1024 ** 3)
ram_available = ram.available / (1024 ** 3)
print(f"RAM: {ram_available:.2f} GB free / {ram_total:.2f} GB total")

In [0]:
%pip install hf_transfer
%restart_python

In [0]:
import docling

## Method 2: Docling Single Document Processing

Docling provides more detailed document analysis including layout detection, table extraction, and image processing.

In [0]:
import datetime
import logging
import time
from pathlib import Path

import numpy as np
from pydantic import TypeAdapter

from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions
from docling.datamodel.base_models import ConversionStatus, InputFormat
from docling.datamodel.pipeline_options import (
    ThreadedPdfPipelineOptions,
    PdfPipelineOptions
)
from docling.datamodel.rapid_ocr_options import RapidOcrOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.pipeline.threaded_standard_pdf_pipeline import ThreadedStandardPdfPipeline
from docling.utils.profiling import ProfilingItem

_log = logging.getLogger(__name__)

logging.getLogger("docling").setLevel(logging.WARNING)
_log.setLevel(logging.INFO)

from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions

# Configure accelerator options for GPU
accelerator_options = AcceleratorOptions(
    device=AcceleratorDevice.CUDA,  # or AcceleratorDevice.AUTO
)

pipeline_options = PdfPipelineOptions()
pipeline_options.ocr_options = RapidOcrOptions(
    backend="torch",
)

doc_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)


In [0]:
doc_converter.initialize_pipeline(InputFormat.PDF)

In [0]:
start_time = time.time()
doc_converter.initialize_pipeline(InputFormat.PDF)
init_runtime = time.time() - start_time
_log.info(f"Pipeline initialized in {init_runtime:.2f} seconds.")

start_time = time.time()
conv_result = doc_converter.convert(input_doc_path)
pipeline_runtime = time.time() - start_time
assert conv_result.status == ConversionStatus.SUCCESS

num_pages = len(conv_result.pages)
_log.info(f"Document converted in {pipeline_runtime:.2f} seconds.")
_log.info(f"  {num_pages / pipeline_runtime:.2f} pages/second.")



In [0]:
# Docling processing test
from docling.document_converter import DocumentConverter

# GPU Support
"https://docling-project.github.io/docling/usage/gpu #achieving-optimal-gpu-performance-with-docling"
from docling.datamodel.accelerator_options import AcceleratorDevice, AcceleratorOptions

# Configure accelerator options for GPU
accelerator_options = AcceleratorOptions(
    device=AcceleratorDevice.CUDA,  # or AcceleratorDevice.AUTO
)

from docling.datamodel.pipeline_options import (
    ThreadedPdfPipelineOptions,
)

pipeline_options = PdfPipelineOptions()
pipeline_options.ocr_options = RapidOcrOptions(
    backend="torch",
)

AcceleratorOptions(device=AcceleratorDevice.CPU)
converter = DocumentConverter(
  
)

file_path = '/Workspace/Users/scott.mckean@databricks.com/multimodal-accelerator/tests/data/wiring_bonding.pdf'

result = converter.convert(file_path)
print(result.status)

In [0]:
import torch, docling, sys
print("Python:", sys.version)
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())

In [0]:
source = "https://arxiv.org/pdf/2408.09869"

converter = DocumentConverter(
  options=None
)
result = converter.convert(source)

# Print Markdown to stdout.
print(result.document.export_to_markdown())

In [0]:
if available_docs:
    # Process first document with Docling
    sample_doc = available_docs[0] 
    print(f"\n🔍 Processing with Docling: {sample_doc.name}")
    
    start_time = time.time()
    
    # Convert document
    result = converter.convert(source=sample_doc)
    document = result.document
    
    processing_time = time.time() - start_time
    
    # Extract information
    pages = len(document.pages)
    pictures = len(document.pictures) 
    tables = len(document.tables)
    main_text = document.export_to_markdown()
    
    print(f"✅ Docling processing completed in {processing_time:.2f}s")
    print(f"📊 Results:")
    print(f"  📄 Pages: {pages}")
    print(f"  🖼️  Pictures: {pictures}")
    print(f"  📋 Tables: {tables}")
    print(f"  📝 Text length: {len(main_text):,} characters")
    
    # Save outputs
    output_dir = Path(f"/Volumes/{CATALOG}/{SCHEMA}/{PROCESSED_DOCS_VOL}") / sample_doc.stem
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Save as JSON and Markdown
    document.save_as_json(output_dir / "document.json")
    document.save_as_markdown(output_dir / "document.md")
    
    print(f"💾 Saved outputs to: {output_dir}")
    
else:
    print("❌ No documents available for processing")

In [0]:
# Store Docling results in tables
if available_docs and 'document' in locals():
    
    def populate_tables_docling(doc_path, file_name, document, output_location):
        """Populate tables with Docling results."""
        try:
            # Insert document record
            doc_sql = f"""
            INSERT OR REPLACE INTO {DOCUMENTS_TABLE} VALUES (
                '{doc_path}',
                '{file_name}',
                'docling_single',
                {len(document.pages)},
                {len(document.pictures)},
                {len(document.tables)},
                0,
                {len(document.export_to_markdown())},
                false,
                '{output_location}',
                map('do_ocr', 'true', 'do_table_structure', 'true'),
                current_timestamp(),
                current_timestamp(),
                'completed'
            )
            """
            
            w.statement_execution.execute_statement(
                warehouse_id="your_warehouse_id",
                statement=doc_sql,
                wait_timeout="30s"
            )
            
            # Create chunks from pages
            chunk_idx = 0
            for page_idx, page in enumerate(document.pages):
                page_text = page.export_to_markdown()
                
                # Split page into manageable chunks
                chunk_size = 1000
                for chunk_start in range(0, len(page_text), chunk_size):
                    chunk_text = page_text[chunk_start:chunk_start + chunk_size]
                    
                    if chunk_text.strip():  # Only insert non-empty chunks
                        chunk_sql = f"""
                        INSERT INTO {CHUNKS_TABLE} VALUES (
                            '{doc_path}',
                            '{file_name}',
                            {chunk_idx},
                            {page_idx + 1},
                            '{chunk_text.replace("'", "''")}',
                            'text',
                            {len(chunk_text.split())},
                            '',
                            '',
                            map('source', 'docling'),
                            current_timestamp()
                        )
                        """
                        
                        w.statement_execution.execute_statement(
                            warehouse_id="your_warehouse_id",
                            statement=chunk_sql,
                            wait_timeout="10s"
                        )
                        
                        chunk_idx += 1
            
            print(f"✅ Populated tables for {file_name} ({chunk_idx} chunks)")
            
        except Exception as e:
            print(f"❌ Error populating tables: {e}")
    
    # Populate tables with Docling results
    populate_tables_docling(
        str(sample_doc), 
        sample_doc.name, 
        document, 
        str(output_dir)
    )

## Processing Summary

View the results of your document processing:

In [0]:
# Query processed documents
try:
    # Get document summary
    docs_query = f"SELECT * FROM {DOCUMENTS_TABLE} ORDER BY created_at DESC LIMIT 10"
    response = w.statement_execution.execute_statement(
        warehouse_id="your_warehouse_id",
        statement=docs_query,
        wait_timeout="30s"
    )
    
    if response.result and response.result.data_array:
        print("📊 Recently Processed Documents:")
        print("-" * 80)
        for row in response.result.data_array:
            file_name = row[1]
            method = row[2] 
            pages = row[3]
            pictures = row[4]
            tables = row[5]
            text_len = row[7]
            status = row[13]
            
            print(f"📄 {file_name}")
            print(f"   Method: {method}")
            print(f"   Content: {pages} pages, {pictures} pictures, {tables} tables")
            print(f"   Text: {text_len:,} characters")
            print(f"   Status: {status}")
            print()
    
    # Get chunk summary
    chunks_query = f"SELECT COUNT(*) as total_chunks, SUM(token_count) as total_tokens FROM {CHUNKS_TABLE}"
    response = w.statement_execution.execute_statement(
        warehouse_id="your_warehouse_id",
        statement=chunks_query,
        wait_timeout="30s"
    )
    
    if response.result and response.result.data_array:
        row = response.result.data_array[0]
        total_chunks = row[0]
        total_tokens = row[1]
        
        print(f"📈 Processing Statistics:")
        print(f"   Total chunks: {total_chunks:,}")
        print(f"   Total tokens: {total_tokens:,}")
        
except Exception as e:
    print(f"❌ Could not query results: {e}")
    print("Make sure to update 'your_warehouse_id' with your actual warehouse ID")

print("\n🎉 Document parsing complete!")
print("\n### Next Steps:")
print("- Run **3_parse_ray.ipynb** for parallel processing")
print("- Run **app.py** to chat with your processed documents")
print("- Check the Delta tables for your processed content")